ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [ ]:
df = pd.read_csv('CC_GENERAL.csv')

**Check the first five rows of the dataset.**

In [ ]:
df.head()

**Check the shape of the dataset.**

In [ ]:
df.shape

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df = df.drop('CUST_ID', axis=1)
df.head()

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df = df.fillna(df.mean())

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(16, 12), bins=20)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['BALANCE'], df['PURCHASES'], alpha=0.5)
plt.xlabel('BALANCE')
plt.ylabel('PURCHASES')
plt.title('BALANCE vs PURCHASES')
plt.show()

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df['BALANCE'], df['CASH_ADVANCE'], alpha=0.5)
plt.xlabel('BALANCE')
plt.ylabel('CASH_ADVANCE')
plt.title('BALANCE vs CASH_ADVANCE')
plt.show()

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

X_scaled[:5]

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

K_range = range(1, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)

inertia_values

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia_values, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Choosing K')
plt.xticks(K_range)
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

K_range = range(2, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

silhouette_scores

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(range(2, 11))
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
score_table = pd.DataFrame({
    'K': list(range(2, 11)),
    'Silhouette Score': silhouette_scores
})

score_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
# Based on the elbow curve (the bend appears around K=3) and the silhouette
# score (which is highest at K=3), we choose K = 3 as the final number of clusters.

final_model = KMeans(n_clusters=3, random_state=42, n_init=10)
final_model.fit(X_scaled)

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df['Cluster'] = final_model.labels_

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby('Cluster').mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df['Cluster'].value_counts().sort_index()

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster'], cmap='viridis', alpha=0.6)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Customer Segments Visualized with PCA')
plt.colorbar(label='Cluster')
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

**1. Why is this an unsupervised learning problem?**

The dataset does not contain any target label that identifies the customer group in advance. We are not predicting a known output. Instead, we are trying to discover hidden patterns and group similar customers together based on their behavior. Since there is no labeled target variable, this is an unsupervised learning problem.

**2. Why did we remove the `CUST_ID` column?**

`CUST_ID` is only an identifier for each customer. It does not describe any behavior such as spending, balance, or payments. Keeping it would add noise to the distance calculation in K-Means and would not help the algorithm find meaningful groups. So we removed it before scaling and clustering.

**3. Which columns had missing values?**

Two columns had missing values:
- `CREDIT_LIMIT` had 1 missing value.
- `MINIMUM_PAYMENTS` had 313 missing values.

**4. How did you handle the missing values?**

We used **mean imputation**, replacing each missing value with the mean of its column using `df.fillna(df.mean())`. This keeps all 8,950 rows in the dataset instead of losing information by dropping them.

**5. Why is scaling important before applying K-Means?**

K-Means uses Euclidean distance to group similar points. The features in this dataset have very different ranges (for example, `CREDIT_LIMIT` can reach thousands while frequency columns are between 0 and 1). Without scaling, the large-range features would dominate the distance calculation and the clustering result would be biased toward them. Using `StandardScaler` puts every feature on the same scale, so each feature contributes fairly to the clusters.

**6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.**

I chose **K = 3**.

- **Elbow method:** In the elbow curve, the inertia drops sharply from K=1 to K=3, and after K=3 the decrease becomes smaller and more gradual. This suggests that K=3 is around the "elbow" of the curve.
- **Silhouette score:** Among the tested K values, the silhouette score is highest at K=3 (about 0.25), which means the clusters are most clearly separated at this value.
- **Business meaning:** K=3 also produces three clearly distinguishable customer segments that are easy to interpret and act on, which is exactly what we want from customer segmentation.

Both methods agreed on K=3, so this is the chosen value.

**7. Based on the cluster summary table, describe each customer segment in your own words.**

Looking at the mean values of each feature per cluster:

- **Cluster 0 — Cash advance users:** These customers have a high balance and very high `CASH_ADVANCE` and `CASH_ADVANCE_FREQUENCY`, but low purchases. They mostly use their credit card to take out cash rather than to buy items. Their `PRC_FULL_PAYMENT` is very low, meaning they rarely pay their balance in full.

- **Cluster 1 — High-value purchasers:** These customers have the highest `PURCHASES`, `ONEOFF_PURCHASES`, `INSTALLMENTS_PURCHASES`, `CREDIT_LIMIT`, and `PAYMENTS`. They also have the highest `PURCHASES_FREQUENCY` and the highest `PRC_FULL_PAYMENT`. They are active buyers who use the card heavily for purchases and pay back well.

- **Cluster 2 — Low-activity / average customers:** This is the largest group. They have low balance, low purchases, low cash advance, and a smaller credit limit. They use the card occasionally but are not very engaged in either purchasing or cash advances.

**8. Which cluster may represent high-value customers?**

**Cluster 1**. They have the highest purchase amounts, the highest credit limit, the highest payments, and the highest full-payment percentage. They are the most active and most reliable spenders, which makes them the most valuable customers for the company.

**9. Which cluster may represent customers who rely more on cash advance?**

**Cluster 0**. These customers have the highest `CASH_ADVANCE` amount and the highest `CASH_ADVANCE_FREQUENCY`, while their purchase activity is low. This is a clear sign that they rely on cash advances much more than the other groups.

**10. How can a company use these clusters for marketing strategy?**

The company can design a different strategy for each segment:

- **Cluster 1 (high-value purchasers):** Reward them with loyalty programs, premium card upgrades, cashback offers, and exclusive deals to keep them engaged and protect their value.
- **Cluster 0 (cash advance users):** Offer financial guidance, lower-interest installment plans, or balance-transfer products to help them reduce reliance on expensive cash advances and improve repayment.
- **Cluster 2 (low-activity customers):** Run activation campaigns such as targeted promotions, introductory discounts, or first-purchase rewards to encourage them to use the card more often.

This way, instead of using one marketing message for everyone, the company can speak to each segment in a way that fits their actual behavior, which increases both customer satisfaction and revenue.